In [2]:
import numpy as np
import pandas as pd

In [10]:
def generate_paths(S0, r, sigma, T, M, N):
    L = len(S0)
    dt = T / M
    # Generate normal variables for each asset, each time step and each path.
    Z = np.random.randn(L, M, N)
    # Compute log-increments for each asset.
    increments = (r[:, np.newaxis, np.newaxis] - 0.5 * sigma[:, np.newaxis, np.newaxis]**2) * dt \
                + sigma[:, np.newaxis, np.newaxis] * np.sqrt(dt) * Z
    # Cumulative sum gives log prices. Prepend zeros so that time0 is S0.
    logS = np.concatenate((np.zeros((L, 1, N)), np.cumsum(increments, axis=1)), axis=1)
    # Exponentiate and scale by S0.
    paths = S0[:, np.newaxis, np.newaxis] * np.exp(logS)
    return paths

def backward_induction(paths, K, r, T, M, degree=2):
    L, num_steps, N = paths.shape  # num_steps = M+1
    dt = T / M
    prices = np.zeros(L)

    # Loop over each asset separately.
    for l in range(L):
        asset_paths = paths[l, :, :]  # shape (M+1, N) for asset l
        # One-period discount factor for this asset.
        disc = np.exp(-r[l] * dt)
        # Compute payoff at maturity for asset l.
        cashflow = np.maximum(K[l] - asset_paths[-1, :], 0)
        
        # Roll backward from t = M-1 down to 1.
        for t in range(M-1, 0, -1):
            # Immediate exercise payoff at time t for asset l.
            immediate = np.maximum(K[l] - asset_paths[t, :], 0)
            # Compute discounted cashflow from later time steps.
            discounted_cf = disc * cashflow
            
            # Identify simulation paths that are in the money.
            itm = immediate > 0
            if np.any(itm):
                X = asset_paths[t, :][itm]  # Underlying asset prices (ITM only)
                Y = discounted_cf[itm]       # Discounted continuation values
                if len(X) > degree:
                    poly = np.polyfit(X, Y, degree)
                    cont_estimate = np.polyval(poly, X)
                else:
                    # Not enough points: assume zero continuation value.
                    cont_estimate = np.zeros_like(X)
                # Compare immediate payoff with estimated continuation value.
                exercise = immediate[itm] > cont_estimate
                indices = np.where(itm)[0]
                # For paths where early exercise is optimal, update cashflow.
                cashflow[indices[exercise]] = immediate[indices[exercise]]
                # For the others, keep the discounted continuation value.
                cashflow[indices[~exercise]] = discounted_cf[indices[~exercise]]
            # Discount cashflow one period backward.
            cashflow = disc * cashflow
        
        # Finally discount one period from time 1 to time 0.
        price_asset = np.mean(cashflow) * np.exp(-r[l] * dt)
        prices[l] = price_asset
        
    return prices

def american_put_lsm(S0, K, r, sigma, T, M=50, N=100000, degree=2):
    K = np.array(K)
    paths = generate_paths(S0, r, sigma, T, M, N)
    return backward_induction(paths, K, r, T, M, degree)

# Example Usage
S0 = np.array([100, 200])  # Initial stock price for a single asset
K = [100, 190]  # Strike price
r = np.array([0.05, 0.05])  # Risk-free rate (annualized)
sigma = np.array([0.2, 0.2])  # Volatility (annualized)
T = 1  # Time to maturity (in years)
M = 50  # Number of time steps
N = 100000  # Number of simulated paths
degree = 2  # Polynomial degree for regression
price = american_put_lsm(S0, K, r, sigma, T)
print(f"LSM American Put Prices: ", price)


LSM American Put Prices:  [5.95400033 7.88175011]


In [ ]:
def generate_paths(S0, r, sigma, T, M, N):
    dt = T / M
    Z = np.random.randn(M, N)  # Normal random variables
    increments = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    log_paths = np.cumsum(increments, axis=0)
    paths = S0 * np.exp(log_paths)
    paths = np.vstack([np.full(N, S0), paths])  # Add initial stock price
    return paths

def backward_induction(paths, K, r, T, M, degree=2):
    dt = T / M
    discount = np.exp(-r * dt)
    cashflows = np.maximum(K - paths[-1], 0)  # Final payoffs at maturity

    for t in range(M - 1, 0, -1):  # Work backwards
        in_the_money = np.where(paths[t] < K)[0]  # Find ITM paths
        if len(in_the_money) == 0:
            continue

        X = paths[t, in_the_money]  # Stock prices at time t
        Y = cashflows[in_the_money] * discount  # Discounted future payoffs

        # Polynomial regression for continuation value
        poly = np.polyfit(X, Y, degree)
        continuation_value = np.polyval(poly, X)

        # Early exercise decision
        exercise_value = K - X
        exercise = exercise_value > continuation_value
        cashflows[in_the_money[exercise]] = exercise_value[exercise]

    return np.mean(cashflows) * discount  # Discount to present value

def american_put_lsm(S0, K, r, sigma, T, M=50, N=100000, degree=2):
    paths = generate_paths(S0, r, sigma, T, M, N)
    return backward_induction(paths, K, r, T, M, degree)

# Example Usage
S0, K, r, sigma, T = 100, 100, 0.05, 0.2, 1
price = american_put_lsm(S0, K, r, sigma, T)
print(f"LSM American Put Price: {price:.4f}")


In [ ]:
df = pd.read_parquet("data/2024_XIU.parquet")

In [14]:
np.cumsum([[[1, 2, 3],
            [1, 4, 3],
            [1, 5, 6]],
            [[2, 4, 5],
            [2, 7, 1],
            [2, 1, 3]]], axis=1)
np.cumsum([[1, 2, 3],
            [1, 4, 3],
            [1, 5, 6]], axis=0)
randoms = np.random.randn(10, 4)
np.vstack([np.full(4, 1), randoms])  # Add initial stock price
np.full(10, 200)[:, np.newaxis, np.newaxis] * np.ones((10, 10, 4))

array([[[200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.]],

       [[200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.]],

       [[200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [200., 200., 200., 200.],
        [2